# NSE Daily Market Data Downloader

Idempotent Colab downloader for NSE Cash Market (CM) daily bhavcopy data.

### What this notebook does

- Mounts Google Drive.
- Uses the project root `/content/drive/MyDrive/quant`.
- Downloads missing NSE CM trading-day files from 2011 through the latest requested date.
- Uses the legacy NSE CM Bhavcopy format through **2024-07-05**.
- Uses the NSE CM UDiFF format from **2024-07-08** onward.
- Never redownloads a valid existing Parquet file.
- Detects and repairs invalid/corrupt existing Parquet files.
- Treats NSE 404s as expected missing-market dates such as weekends/holidays.
- Writes Parquet atomically through temporary files.
- Maintains a CSV manifest.
- Validates the resulting dataset and prints a run summary.

> **Important:** NSE can change archive URLs and dissemination conventions. If a current UDiFF URL stops working, update the URL builder in the configuration/helper section rather than changing the rest of the pipeline.


## 1. Setup

In [ ]:
# Install runtime dependencies.
# Run this cell once per fresh Colab runtime.

!pip -q install pandas pyarrow requests duckdb tqdm


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
from datetime import date, datetime, timedelta
import io
import os
import time
import zipfile
import hashlib
import logging

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

print("Environment ready.")


## 2. Configuration

In [ ]:
# ---------------------------------------------------------------------
# Project / storage configuration
# ---------------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/quant")
DATA_DIR = BASE_DIR / "data"
PARQUET_DIR = DATA_DIR / "parquet"
RAW_DIR = DATA_DIR / "raw" / "nse_bhavcopy"
METADATA_DIR = DATA_DIR / "metadata"
MANIFEST_FILE = DATA_DIR / "download_manifest.csv"

# ---------------------------------------------------------------------
# Date range
# ---------------------------------------------------------------------

START_DATE = date(2011, 1, 1)

# None = automatically use today's date in the Colab runtime.
END_DATE = None

# NSE CM format transition.
LEGACY_END_DATE = date(2024, 7, 5)
UDIFF_START_DATE = date(2024, 7, 8)

# ---------------------------------------------------------------------
# Optional symbol filter
# ---------------------------------------------------------------------

# None = keep every security in the NSE CM bhavcopy.
# Example: STOCKS = {"RELIANCE", "TCS", "INFY"}
STOCKS = None

# ---------------------------------------------------------------------
# Download behavior
# ---------------------------------------------------------------------

REQUEST_TIMEOUT = 60
MAX_RETRIES = 5
BACKOFF_FACTOR = 1.5
SLEEP_BETWEEN_REQUESTS = 0.10

# If True, an existing invalid/corrupt Parquet is deleted and downloaded again.
REPAIR_INVALID_FILES = True

# If True, successfully downloaded CSV files are also retained under RAW_DIR.
# Keeping raw files consumes substantially more Drive space.
KEEP_RAW_ZIPS = False

# ---------------------------------------------------------------------
# Output columns
# ---------------------------------------------------------------------

EXPECTED_COLUMNS = [
    "date",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "volume",
]

BASE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

if END_DATE is None:
    END_DATE = date.today()

assert START_DATE <= END_DATE, "START_DATE must be <= END_DATE"

print("BASE_DIR :", BASE_DIR)
print("DATA_DIR :", DATA_DIR)
print("PARQUET_DIR:", PARQUET_DIR)
print("Range    :", START_DATE, "to", END_DATE)


In [ ]:
# Search for existing Git repositories
git_dirs = []

for root, dirs, files in os.walk("/content"):
    if ".git" in dirs:
        git_dirs.append(Path(root))
        dirs[:] = []  # don't recurse further into .git

print("Git repositories found:")

for repo in git_dirs:
    print(" -", repo)

In [ ]:
# Try common locations first
possible_repos = [
    Path("/content/quant")
]

REPO_DIR = None

for p in possible_repos:
    if (p / ".git").exists():
        REPO_DIR = p
        break

# Otherwise use the first discovered repository
if REPO_DIR is None and git_dirs:
    REPO_DIR = git_dirs[0]

if REPO_DIR is None:
    raise RuntimeError(
        "Could not find the existing Git repository under /content."
    )

print("Using repository:")
print(REPO_DIR)

In [ ]:
%cd $REPO_DIR

print("Current branch:")
!git branch --show-current

print("\nGit status:")
!git status --short

print("\nLatest commit:")
!git log -1 --oneline

In [ ]:
TARGET_BRANCH = "feature/quant-research-framework"

current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=REPO_DIR
).decode().strip()

print("Current branch:", current_branch)

if current_branch != TARGET_BRANCH:
    print(f"Switching to {TARGET_BRANCH}...")
    
    result = subprocess.run(
        ["git", "checkout", TARGET_BRANCH],
        cwd=REPO_DIR,
        capture_output=True,
        text=True
    )
    
    print(result.stdout)
    print(result.stderr)

print("\nActive branch:")
subprocess.run(["git", "branch", "--show-current"], cwd=REPO_DIR)

## 3. Logging and HTTP session

In [ ]:
# NSE requests benefit from a realistic browser-like session and retries.

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("nse_downloader")

session = requests.Session()

retry = Retry(
    total=MAX_RETRIES,
    connect=MAX_RETRIES,
    read=MAX_RETRIES,
    status=MAX_RETRIES,
    backoff_factor=BACKOFF_FACTOR,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/",
    "Connection": "keep-alive",
})

print("HTTP session configured.")


## 4. Helper functions — dates, paths, and URLs

In [ ]:
def trading_day_candidates(start_date: date, end_date: date):
    """Yield weekdays. NSE holidays are filtered later by HTTP 404 / validation."""
    current = start_date
    while current <= end_date:
        if current.weekday() < 5:
            yield current
        current += timedelta(days=1)


def parquet_path_for_day(day: date) -> Path:
    """Return Hive-style yearly output path."""
    year_dir = PARQUET_DIR / f"year={day.year}"
    year_dir.mkdir(parents=True, exist_ok=True)
    return year_dir / f"nse_cm_{day:%Y%m%d}.parquet"


def raw_zip_path_for_day(day: date) -> Path:
    return RAW_DIR / f"nse_cm_{day:%Y%m%d}.zip"


def legacy_url(day: date) -> str:
    """NSE legacy CM Bhavcopy URL used before the UDiFF transition."""
    month = day.strftime("%b").upper()
    filename = f"cm{day:%d}{month}{day:%Y}bhav.csv.zip"
    return (
        "https://nsearchives.nseindia.com/content/"
        f"historical/EQUITIES/{day.year}/{month}/{filename}"
    )


def udiff_url(day: date) -> str:
    """NSE CM UDiFF Common Bhavcopy Final URL."""
    filename = f"BhavCopy_NSE_CM_0_0_0_{day:%Y%m%d}_F_0000.csv.zip"
    return f"https://nsearchives.nseindia.com/content/cm/{filename}"


def url_for_day(day: date) -> str:
    if day <= LEGACY_END_DATE:
        return legacy_url(day)
    return udiff_url(day)


# Quick examples
for d in [date(2011, 1, 3), date(2024, 7, 5), date(2024, 7, 8)]:
    print(d, "->", url_for_day(d))


## 5. Helper functions — parsing and normalization

In [ ]:
def normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace(".", "_")
        .replace("/", "_")
    )


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize common NSE column-name variants to the project schema."""
    rename = {}

    for col in df.columns:
        n = normalize_column_name(col)

        aliases = {
            "tradingsymbol": "symbol",
            "symbol": "symbol",
            "timestamp": "date",
            "trade_date": "date",
            "date": "date",
            "open_price": "open",
            "open": "open",
            "high_price": "high",
            "high": "high",
            "low_price": "low",
            "low": "low",
            "close_price": "close",
            "close": "close",
            "last_price": "close",
            "ltp": "close",
            "tottrdqty": "volume",
            "total_traded_quantity": "volume",
            "total_traded_qty": "volume",
            "volume": "volume",
        }

        if n in aliases:
            rename[col] = aliases[n]

    df = df.rename(columns=rename)

    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            f"Received columns: {list(df.columns)}"
        )

    df = df[EXPECTED_COLUMNS].copy()

    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date
    df["symbol"] = df["symbol"].astype("string").str.strip()

    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["date", "symbol"])

    if STOCKS is not None:
        df = df[df["symbol"].isin(STOCKS)]

    # Remove accidental duplicate symbol rows.
    df = df.drop_duplicates(subset=["date", "symbol"], keep="last")

    return df


def read_nse_zip(content: bytes) -> pd.DataFrame:
    """Read the first CSV inside an NSE ZIP archive."""
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        csv_names = [
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ]

        if not csv_names:
            raise ValueError("ZIP archive contains no CSV file.")

        with z.open(csv_names[0]) as f:
            df = pd.read_csv(f)

    return normalize_columns(df)


## 6. Helper functions — validation

In [ ]:
def validate_dataframe(df: pd.DataFrame, expected_day: date) -> tuple[bool, str]:
    """Validate an in-memory daily dataframe."""
    if df.empty:
        return False, "empty_dataframe"

    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing:
        return False, f"missing_columns:{missing}"

    if df["date"].isna().any():
        return False, "null_dates"

    if not (df["date"] == expected_day).all():
        bad_dates = sorted(set(df["date"]) - {expected_day})
        return False, f"wrong_date:{bad_dates[:5]}"

    if df["symbol"].isna().any() or (df["symbol"].astype(str).str.len() == 0).any():
        return False, "invalid_symbols"

    numeric_cols = ["open", "high", "low", "close", "volume"]
    if df[numeric_cols].isna().all(axis=1).any():
        return False, "rows_with_all_numeric_values_null"

    if len(df) == 0:
        return False, "zero_rows"

    return True, "ok"


def validate_parquet(path: Path, expected_day: date) -> tuple[bool, str, int]:
    """Read and validate an existing Parquet file."""
    if not path.exists():
        return False, "missing_file", 0

    try:
        table = pq.read_table(path)
        df = table.to_pandas()

        df = normalize_columns(df)
        ok, reason = validate_dataframe(df, expected_day)

        return ok, reason, len(df)

    except Exception as exc:
        return False, f"parquet_read_error:{type(exc).__name__}:{exc}", 0


def atomic_write_parquet(df: pd.DataFrame, path: Path):
    """Write to a temporary file and atomically replace the destination."""
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path.with_suffix(".parquet.tmp")

    if tmp_path.exists():
        tmp_path.unlink()

    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, tmp_path, compression="snappy")

    # Validate before replacing the final file.
    ok, reason, rows = validate_parquet(tmp_path, df["date"].iloc[0])
    if not ok:
        tmp_path.unlink(missing_ok=True)
        raise ValueError(f"Post-write validation failed: {reason}")

    tmp_path.replace(path)

    return rows


## 7. Helper functions — manifest

In [ ]:
MANIFEST_COLUMNS = [
    "date",
    "status",
    "rows",
    "path",
    "url",
    "message",
    "checked_at",
]


def load_manifest() -> pd.DataFrame:
    if not MANIFEST_FILE.exists():
        return pd.DataFrame(columns=MANIFEST_COLUMNS)

    try:
        manifest = pd.read_csv(MANIFEST_FILE)
        for col in MANIFEST_COLUMNS:
            if col not in manifest.columns:
                manifest[col] = None
        return manifest[MANIFEST_COLUMNS]
    except Exception as exc:
        logger.warning("Could not read manifest: %s", exc)
        return pd.DataFrame(columns=MANIFEST_COLUMNS)


manifest = load_manifest()


def append_manifest(record: dict):
    global manifest

    row = {col: record.get(col) for col in MANIFEST_COLUMNS}
    row["checked_at"] = datetime.now().isoformat(timespec="seconds")

    manifest = pd.concat(
        [manifest, pd.DataFrame([row])],
        ignore_index=True,
    )

    # Keep one latest record per date.
    manifest["date"] = manifest["date"].astype(str)
    manifest = (
        manifest
        .drop_duplicates(subset=["date"], keep="last")
        .sort_values("date")
    )

    MANIFEST_FILE.parent.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(MANIFEST_FILE, index=False)


print(f"Manifest records loaded: {len(manifest):,}")


## 8. Helper functions — download one day

In [ ]:
def download_day(day: date) -> dict:
    """Download, parse, validate and persist one NSE CM day."""
    path = parquet_path_for_day(day)
    url = url_for_day(day)

    # Idempotency: valid files are never downloaded again.
    if path.exists():
        ok, reason, rows = validate_parquet(path, day)

        if ok:
            return {
                "date": day.isoformat(),
                "status": "already_valid",
                "rows": rows,
                "path": str(path),
                "url": url,
                "message": "Existing Parquet passed validation.",
            }

        if not REPAIR_INVALID_FILES:
            return {
                "date": day.isoformat(),
                "status": "invalid_skipped",
                "rows": rows,
                "path": str(path),
                "url": url,
                "message": reason,
            }

        logger.warning("%s invalid: %s — repairing.", day, reason)
        path.unlink(missing_ok=True)

    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT)

        # NSE commonly returns 404 for dates without an archive.
        if response.status_code == 404:
            return {
                "date": day.isoformat(),
                "status": "not_found",
                "rows": 0,
                "path": str(path),
                "url": url,
                "message": "NSE archive not found; likely holiday/non-trading day.",
            }

        response.raise_for_status()

        content_type = response.headers.get("Content-Type", "")
        content = response.content

        if not content:
            raise ValueError("NSE returned an empty response.")

        # Save raw ZIP only if requested.
        if KEEP_RAW_ZIPS:
            raw_path = raw_zip_path_for_day(day)
            raw_path.write_bytes(content)

        df = read_nse_zip(content)

        ok, reason = validate_dataframe(df, day)
        if not ok:
            raise ValueError(f"Downloaded data failed validation: {reason}")

        rows = atomic_write_parquet(df, path)

        return {
            "date": day.isoformat(),
            "status": "downloaded",
            "rows": rows,
            "path": str(path),
            "url": url,
            "message": f"Downloaded successfully; content-type={content_type}",
        }

    except Exception as exc:
        return {
            "date": day.isoformat(),
            "status": "error",
            "rows": 0,
            "path": str(path),
            "url": url,
            "message": f"{type(exc).__name__}: {exc}",
        }


## 9. Pre-run inventory

In [ ]:
candidates = list(trading_day_candidates(START_DATE, END_DATE))

existing_valid = 0
existing_invalid = 0
missing = 0

for day in candidates:
    path = parquet_path_for_day(day)

    if not path.exists():
        missing += 1
        continue

    ok, _, _ = validate_parquet(path, day)

    if ok:
        existing_valid += 1
    else:
        existing_invalid += 1

print(f"Weekday candidates : {len(candidates):,}")
print(f"Valid existing     : {existing_valid:,}")
print(f"Invalid existing   : {existing_invalid:,}")
print(f"Missing            : {missing:,}")


## 10. Download loop

In [ ]:
run_started = datetime.now()
run_results = []

for day in tqdm(candidates, desc="NSE daily files"):
    result = download_day(day)
    run_results.append(result)

    append_manifest(result)

    if result["status"] in {"downloaded", "error"}:
        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(SLEEP_BETWEEN_REQUESTS)

run_finished = datetime.now()

run_df = pd.DataFrame(run_results)

print()
print("Run completed.")
print("Started :", run_started)
print("Finished:", run_finished)
print("Elapsed :", run_finished - run_started)


## 11. Run summary

In [ ]:
if run_df.empty:
    print("No dates were processed.")
else:
    print("Status counts:")
    display(
        run_df["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="count")
    )

    print("Rows downloaded:")
    print(f"{run_df.loc[run_df['status'] == 'downloaded', 'rows'].sum():,}")

    errors = run_df[run_df["status"] == "error"]

    if not errors.empty:
        print()
        print("ERRORS:")
        display(errors[["date", "message", "url"]])

    print()
    print(f"Manifest: {MANIFEST_FILE}")


## 12. Dataset validation

In [ ]:
# Validate every Parquet file in the requested date range.

validation_results = []

for day in candidates:
    path = parquet_path_for_day(day)

    if not path.exists():
        validation_results.append({
            "date": day.isoformat(),
            "status": "missing",
            "rows": 0,
            "path": str(path),
            "message": "No Parquet file.",
        })
        continue

    ok, reason, rows = validate_parquet(path, day)

    validation_results.append({
        "date": day.isoformat(),
        "status": "valid" if ok else "invalid",
        "rows": rows,
        "path": str(path),
        "message": reason,
    })

validation_df = pd.DataFrame(validation_results)

display(
    validation_df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

invalid = validation_df[validation_df["status"] == "invalid"]
missing = validation_df[validation_df["status"] == "missing"]

print(f"Invalid files: {len(invalid):,}")
print(f"Missing files: {len(missing):,}")

if not invalid.empty:
    display(invalid)

if not missing.empty:
    print("Missing dates are not automatically errors; NSE holidays/non-trading days are expected.")
    display(missing.head(50))


## 13. Dataset statistics

In [ ]:
# Fast filesystem-level summary without loading the full dataset into memory.

parquet_files = sorted(PARQUET_DIR.glob("year=*/nse_cm_*.parquet"))

file_rows = []

for path in parquet_files:
    try:
        metadata = pq.ParquetFile(path).metadata
        rows = metadata.num_rows
        size_mb = path.stat().st_size / (1024 ** 2)

        file_rows.append({
            "file": str(path),
            "rows": rows,
            "size_mb": round(size_mb, 2),
        })
    except Exception as exc:
        file_rows.append({
            "file": str(path),
            "rows": None,
            "size_mb": None,
            "error": str(exc),
        })

inventory_df = pd.DataFrame(file_rows)

print(f"Parquet files found: {len(inventory_df):,}")

if not inventory_df.empty:
    print(f"Total rows: {inventory_df['rows'].sum():,.0f}")
    print(f"Total size: {inventory_df['size_mb'].sum():,.1f} MB")

    display(
        inventory_df
        .assign(year=inventory_df["file"].str.extract(r"year=(\d{4})")[0])
        .groupby("year", dropna=False)
        .agg(
            files=("file", "count"),
            rows=("rows", "sum"),
            size_mb=("size_mb", "sum"),
        )
        .reset_index()
    )


## 14. Optional DuckDB sanity check

In [ ]:
# Optional: inspect the complete Parquet dataset with DuckDB.
# This is useful before starting backtests.

import duckdb

glob_path = str(PARQUET_DIR / "**" / "*.parquet")

con = duckdb.connect()

summary = con.execute(f'''
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT symbol) AS symbols,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM read_parquet('{glob_path}', hive_partitioning=true)
''').df()

display(summary)

print("Example RELIANCE rows:")

reliance = con.execute(f'''
    SELECT date, symbol, open, high, low, close, volume
    FROM read_parquet('{glob_path}', hive_partitioning=true)
    WHERE symbol = 'RELIANCE'
    ORDER BY date
    LIMIT 10
''').df()

display(reliance)


## 15. How to run this notebook

### First test

Change:

```python
START_DATE = date(2024, 7, 8)
END_DATE = date(2024, 7, 12)
```

Run all cells and inspect the summary/validation output.

### Full backfill

Then change back to:

```python
START_DATE = date(2011, 1, 1)
END_DATE = None
```

The notebook is designed to be rerun safely:

- valid Parquet → **skip**
- missing Parquet → **download**
- invalid/corrupt Parquet → **repair**
- NSE 404 → **record as not_found**
- successful download → **validate before final write**
- manifest → **updated after each processed date**

### Important storage rule

Do **not** put the downloaded market data into GitHub. Keep the code/notebook in your `quant_ai` repository and keep market data/results in Google Drive under:

```text
/content/drive/MyDrive/quant/
```

This keeps the repository lightweight and makes Colab backtesting reproducible.
